# Simple Models on Embeddings

This notebook loads precomputed embeddings from `artifacts/embedding_cnn/embeddings` and trains lightweight classifiers. You can swap models, adjust hyperparameters, and vary the amount of training data to see how accuracy changes.

## 1. Load embeddings

In [ ]:
import numpy as np
import os
from pathlib import Path

emb_dir = Path('artifacts/embedding_cnn/embeddings')
train_file = emb_dir / 'train_embeddings.npz'
test_file = emb_dir / 'test_embeddings.npz'
assert train_file.exists(), train_file
assert test_file.exists(), test_file

data = np.load(train_file, allow_pickle=True)
X_train, y_train = data['embeddings'], data['labels']
data = np.load(test_file, allow_pickle=True)
X_test, y_test = data['embeddings'], data['labels']
print('train', X_train.shape, y_train.shape)
print('test', X_test.shape, y_test.shape)

## 2. Define a training/evaluation helper

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(clf, X_train, y_train, X_test, y_test):
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    return accuracy_score(y_test, y_pred), f1_score(y_test, y_pred, average='macro')

## 3. Try a few default classifiers

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

models = {
    'logreg': LogisticRegression(max_iter=2000),
    'rf': RandomForestClassifier(n_estimators=100),
    'svc': SVC(probability=True),
}

for name, clf in models.items():
    acc, f1 = evaluate(clf, X_train, y_train, X_test, y_test)
    print(f'{name}: acc={acc:.4f}, f1={f1:.4f}')

## 4. Experiment with training size and hyperparameters
Change `n_samples` or the model constructors below and rerun cells.

In [ ]:
# reduce training set size for quick experiments
n_samples = 100  # try 50, 200, len(X_train) etc.
idx = np.random.choice(len(X_train), size=n_samples, replace=False)
X_sub, y_sub = X_train[idx], y_train[idx]

# modify or add your own model here
from sklearn.ensemble import GradientBoostingClassifier
clf = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1)
acc, f1 = evaluate(clf, X_sub, y_sub, X_test, y_test)
print(f'train size={n_samples} -> acc={acc:.4f}, f1={f1:.4f}')

## 5. Using other libraries or custom models
You can install additional packages (e.g. `xgboost`, `lightgbm`) and then import them here. Example below is commented out.

In [ ]:
# !pip install xgboost  # uncomment if needed
# from xgboost import XGBClassifier
# clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
# acc, f1 = evaluate(clf, X_train, y_train, X_test, y_test)
# print('xgboost', acc, f1)

## 6. What next?
- Try cross-validation or grid search for hyperparameters.
- Plot decision boundaries or confusion matrices.
- Use embedding generation code to create new embeddings (e.g. from different networks) and repeat experiments.
- This notebook is deliberately simple so you can modify it freely.